# CRP Model Comparison Report

**Experiment**: `crp-model-comparison`  
**Dataset**: `Completion_prediction__dataset__hashing_500k.parquet` (100k sample)  
**Models**: LightGBM vs XGBoost

---
> Run `make mlflow-experiment` truoc de co data trong experiment.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENT_NAME = 'crp-model-comparison'

# SQLite backend -- must match mlflow_experiment.py
db_path = PROJECT_ROOT / 'mlflow.db'
TRACKING_URI = f'sqlite:///{db_path.as_posix()}'
mlflow.set_tracking_uri(TRACKING_URI)

COLORS = {
    'LightGBM': '#4C9BE8',
    'XGBoost':  '#E87B4C',
}

plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#30363D',
    'axes.labelcolor':  '#C9D1D9',
    'xtick.color':      '#C9D1D9',
    'ytick.color':      '#C9D1D9',
    'text.color':       '#C9D1D9',
    'grid.color':       '#21262D',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'sans-serif',
})

print(f'Tracking URI: {TRACKING_URI}')
print(f'DB exists: {db_path.exists()}')

## 1. Load Runs from MLflow

In [ ]:
try:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        raise ValueError(f'Experiment "{EXPERIMENT_NAME}" not found. Run: make mlflow-experiment')

    runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['metrics.test_auc_roc DESC'],
    )

    # Exclude summary run
    runs_df = runs_df[~runs_df['tags.mlflow.runName'].str.startswith('__', na=False)]

    print(f'Found {len(runs_df)} model runs in "{EXPERIMENT_NAME}"')
    metric_cols_found = [c for c in runs_df.columns if c.startswith('metrics.')]
    print(f'Metrics: {metric_cols_found}')

except Exception as e:
    print(f'ERROR: {e}')
    runs_df = pd.DataFrame()

## 2. Metrics Comparison Table

In [ ]:
if not runs_df.empty:
    want = ['test_auc_roc', 'test_auc_pr', 'test_log_loss', 'test_f1', 'test_ece', 'train_time_seconds']
    avail = [f'metrics.{c}' for c in want if f'metrics.{c}' in runs_df.columns]

    display_df = runs_df[['tags.mlflow.runName'] + avail].copy()
    display_df.columns = ['Model'] + [
        c.replace('metrics.test_', '').replace('metrics.', '').upper() for c in avail
    ]
    display_df = display_df.set_index('Model').sort_values('AUC_ROC', ascending=False)

    def highlight_best(df):
        styles = pd.DataFrame('', index=df.index, columns=df.columns)
        for col in df.columns:
            best_idx = df[col].idxmin() if col in ['LOG_LOSS', 'ECE'] else df[col].idxmax()
            styles.loc[best_idx, col] = 'background-color: #1f6e2e; font-weight: bold; color: white'
        return styles

    fmt = {col: '{:.4f}' for col in display_df.columns if col != 'TRAIN_TIME_SECONDS'}
    styled = (
        display_df.style
        .apply(highlight_best, axis=None)
        .format(fmt)
        .set_caption('Model Comparison -- Test Set (green = best)')
    )
    display(styled)
    best_model = display_df['AUC_ROC'].idxmax()
    print(f'Best model: {best_model}  (AUC-ROC = {display_df["AUC_ROC"].max():.4f})')

## 3. Metrics Bar Chart

In [ ]:
if not runs_df.empty:
    metrics_to_plot = [m for m in ['AUC_ROC', 'AUC_PR', 'F1'] if m in display_df.columns]

    fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(12, 5))
    if len(metrics_to_plot) == 1:
        axes = [axes]
    fig.suptitle('CRP Model Comparison -- Test Set Metrics', fontsize=14, fontweight='bold', color='#E6EDF3')

    for ax, metric in zip(axes, metrics_to_plot):
        vals = display_df[metric].sort_values(ascending=True)
        colors = [COLORS.get(m, '#888') for m in vals.index]
        bars = ax.barh(vals.index, vals.values, color=colors, edgecolor='#30363D', linewidth=0.5)

        for bar, val in zip(bars, vals.values):
            ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
                    f'{val:.4f}', va='center', fontsize=9, color='#C9D1D9')

        ax.set_title(metric.replace('_', '-'), fontsize=11, color='#E6EDF3')
        ax.set_xlim(vals.min() * 0.97, vals.max() * 1.03)
        ax.grid(axis='x')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    out = PROJECT_ROOT / 'data/models'
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / 'metrics_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
    plt.show()
    print('Saved: data/models/metrics_comparison.png')

## 4. Training Time vs AUC-ROC

In [ ]:
if not runs_df.empty and 'TRAIN_TIME_SECONDS' in display_df.columns:
    fig, ax = plt.subplots(figsize=(7, 5))

    for model_name, row in display_df.iterrows():
        color = COLORS.get(model_name, '#888')
        ax.scatter(row['TRAIN_TIME_SECONDS'], row['AUC_ROC'],
                   s=250, color=color, zorder=5, edgecolors='white', linewidths=0.8)
        ax.annotate(model_name, (row['TRAIN_TIME_SECONDS'], row['AUC_ROC']),
                    textcoords='offset points', xytext=(10, 4), fontsize=10, color=color)

    ax.set_xlabel('Training Time (seconds)', fontsize=11)
    ax.set_ylabel('AUC-ROC (Test)', fontsize=11)
    ax.set_title('Training Efficiency', fontsize=13, fontweight='bold', color='#E6EDF3')
    ax.grid(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    fig.savefig(PROJECT_ROOT / 'data/models/efficiency_comparison.png', dpi=150,
                bbox_inches='tight', facecolor='#0D1117')
    plt.show()

## 5. MLflow Run Details

In [ ]:
if not runs_df.empty:
    print('MLflow Run Details\n' + '='*60)
    for _, run in runs_df.iterrows():
        name = run.get('tags.mlflow.runName', 'N/A')
        rid = run['run_id']
        auc = run.get('metrics.test_auc_roc', float('nan'))
        uri = run['artifact_uri']
        print(f'  {name:<16} | run_id: {rid[:8]}... | AUC-ROC: {auc:.4f}')
        print(f'  {"":<16}   artifact_uri: {uri}')
        print()
    print('Open MLflow UI: http://127.0.0.1:5000  (run: make mlflow-ui)')

## 6. Export HTML Report

In [ ]:
import subprocess
report_path = PROJECT_ROOT / 'data/models/model_comparison_report.html'
result = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', '--no-input',
     '--output', str(report_path),
     str(PROJECT_ROOT / 'notebooks/mlflow_model_comparison.ipynb')],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'HTML report exported -> {report_path}')
else:
    print(f'Error: {result.stderr}')